In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   22G  205G  10% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G     0  5.8G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1       233G   23G  210G  10% /kaggle/input
tmpfs           6.4G   36K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

GZ_DB = Path("/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz")

OUT_DIR = Path("/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", GZ_DB)
print("Output:", OUT_DIR)

ValueError: Mountpoint must not already contain files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import shutil, struct

GZ_DB = Path("/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz")

def gzip_isize(gz_path: Path):
    """
    Ambil ISIZE (uncompressed size mod 2^32) dari footer gzip.
    Catatan: kalau >4GB, nilainya bisa wrap (mod 4GB).
    Tetap berguna sebagai perkiraan minimum.
    """
    with open(gz_path, "rb") as f:
        f.seek(-4, 2)
        return struct.unpack("<I", f.read(4))[0]

def free_space_bytes(path: str):
    usage = shutil.disk_usage(path)
    return usage.free

need = gzip_isize(GZ_DB)
print("Perkiraan minimum ukuran .db hasil ekstrak (bytes):", need)
print("≈ (GB):", need/1e9)

paths_to_check = ["/content", "/dev/shm", "/content/drive"]
for p in paths_to_check:
    try:
        free = free_space_bytes(p)
        print(f"Free space {p}: {free/1e9:.2f} GB")
    except Exception as e:
        print(f"Gagal cek {p}:", e)

Perkiraan minimum ukuran .db hasil ekstrak (bytes): 3287887872
≈ (GB): 3.287887872
Free space /content: 219.61 GB
Free space /dev/shm: 6.13 GB
Free space /content/drive: 113.71 GB


In [ ]:
# ============================================================
# CELL 1: Setup
# ============================================================
from pathlib import Path

GZ_DB   = Path("/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz")
OUT_DIR = Path("/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV_GZ = OUT_DIR / "ILtrain1_perovskite.csv.gz"

# ⚡ DB tetap di /tmp lokal (wajib, SQLite butuh full file)
DB_PATH = Path("/tmp/ILtrain1_tmp.db")

print("Input :", GZ_DB)
print("Output:", OUT_CSV_GZ)

Input : /content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz
Output: /content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1_perovskite.csv.gz


In [ ]:
# ============================================================
# CELL 2: Ekstrak + langsung query + tulis CSV — semua pipeline
#         Hapus DB segera setelah selesai (hemat disk)
# ============================================================
import gzip, shutil, sqlite3, csv, gc
from pathlib import Path

# --- Ekstrak ke /tmp ---
if DB_PATH.exists():
    DB_PATH.unlink()

print("📦 Ekstrak .gz → /tmp ...")
with gzip.open(GZ_DB, "rb") as f_in, open(DB_PATH, "wb") as f_out:
    shutil.copyfileobj(f_in, f_out, length=4 * 1024 * 1024)
print(f"   DB size: {DB_PATH.stat().st_size/1e9:.2f} GB")

# --- Deteksi tabel & kolom ---
conn = sqlite3.connect(str(DB_PATH))
conn.execute("PRAGMA cache_size = -32000")
conn.execute("PRAGMA temp_store = MEMORY")

MAIN_TABLE = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name LIMIT 1;"
).fetchone()[0]
cols = [r[1] for r in conn.execute(f"PRAGMA table_info('{MAIN_TABLE}');")]
print(f"✅ Tabel: {MAIN_TABLE}")
print(f"   Kolom: {cols}")

TARGET_COL = "material_component"  # ← GANTI sesuai kolom Anda

# --- Streaming query → langsung CSV.GZ (tanpa pandas, lebih hemat RAM) ---
query = f'''
    SELECT * FROM "{MAIN_TABLE}"
    WHERE LOWER(COALESCE("{TARGET_COL}", "")) LIKE "%perovskite%"
'''

CHUNKSIZE  = 10_000
rows_total = 0

print(f"\n🚀 Streaming langsung → {OUT_CSV_GZ} ...")
with gzip.open(OUT_CSV_GZ, "wt", encoding="utf-8", newline="") as gz_out:
    writer = csv.writer(gz_out)
    writer.writerow(cols)  # header

    cursor = conn.cursor()
    cursor.execute(query)

    while True:
        rows = cursor.fetchmany(CHUNKSIZE)  # ⚡ ambil sebagian, bukan semua
        if not rows:
            break
        writer.writerows(rows)
        rows_total += len(rows)
        print(f"  ✔ {rows_total:,} baris ditulis...")
        del rows
        gc.collect()

cursor.close()
conn.close()

print(f"\n✅ Selesai! Total baris : {rows_total:,}")
print(f"   Output size         : {OUT_CSV_GZ.stat().st_size/1e6:.2f} MB")

# ⚡ Hapus DB temp SEGERA setelah selesai
DB_PATH.unlink()
print(f"🗑️  Temp DB dihapus otomatis.")

📦 Ekstrak .gz → /tmp ...


OSError: [Errno 28] No space left on device